# 5.3 动态行数：让同一个kernel处理不同大小的输入

5.2用Cube完成矩阵乘，再用Vector加偏置。本节继续计算`Y = A @ B.T + Bias`，以动态行数 M 为例，学习怎样让同一个编译好的 kernel 处理形状变化的输入。

考虑模型中一个固定的线性层：每一行输入表示一个样本或一个 token 的特征。输入特征数 K、输出特征数 N 由该层权重矩阵的形状确定，保持不变；每次送入的样本数或 token 数可能不同，因此输入 A 的行数 M 会变化。本节据此固定 B 和 Bias 的形状，只把 M 改为运行时确定。

局部空间分配、Cube矩阵乘和Vector加偏置都沿用5.2。本节重点看动态M带来的变化：实际行数怎样传入kernel，怎样决定任务数量和访问范围，调用前要检查什么，以及怎样复用同一个编译结果。

本节的重点是**哪些内容随M变化，哪些内容保持不变**。

<small>线上环境没有昇腾950，可直接阅读本页代码和实测结果；自行运行请按1.4准备环境，并以本章目录为工作目录。Notebook用户从下方准备单元开始顺序执行。</small>

<small>下方辅助代码与5.2相同，阅读时可先看第1部分。</small>

In [1]:
from pathlib import Path
import re
import subprocess
import sys

Path("src").mkdir(exist_ok=True)
Path("answer").mkdir(exist_ok=True)

def run_and_show(command):
    # NPU程序在独立进程中运行；当前Notebook内核不导入torch_npu。
    result = subprocess.run(
        command, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    )
    # 只省略安装目录；数值、校验结果和报错内容保持原样。
    output = result.stdout.replace(str(Path.cwd()), ".")
    output = re.sub(r"/home/[^\s]+", "<已省略安装路径>", output)
    print(output, end="")
    result.check_returncode()
    return result


## 1. 从5.2出发：只让M变化

在5.2中，A和Y都有128行。现在假设连续收到三批输入，A分别有64、128和192行，B和Bias保持不变。我们希望先编译一次，再用得到的同一个kernel对象处理这三批输入。

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;">对象</th>
<th style="text-align: left; vertical-align: top;">形状或取值</th>
<th style="text-align: left; vertical-align: top;">本节是否变化</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;"><code>A</code></td>
<td style="text-align: left; vertical-align: top;"><code>(M, 128)</code>，float16</td>
<td style="text-align: left; vertical-align: top;">行数M变化</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>B</code></td>
<td style="text-align: left; vertical-align: top;"><code>(128, 128)</code>，float16</td>
<td style="text-align: left; vertical-align: top;">保持不变</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>Bias</code></td>
<td style="text-align: left; vertical-align: top;"><code>(128,)</code>，float32</td>
<td style="text-align: left; vertical-align: top;">保持不变</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>Y</code></td>
<td style="text-align: left; vertical-align: top;"><code>(M, 128)</code>，float32</td>
<td style="text-align: left; vertical-align: top;">随A改变行数</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">局部块</td>
<td style="text-align: left; vertical-align: top;"><code>BM = BN = BK = 64</code></td>
<td style="text-align: left; vertical-align: top;">保持不变</td>
</tr>
</tbody>
</table>

每个任务仍计算一个64×64的Y结果块。Y有128列，所以沿列方向始终分成两块；沿行方向的块数则由M决定。

![不同M对应的结果块与固定局部空间](images/05_03_dynamic_tiles.png)

<small>图5-3　M变化时，任务数量变化，局部块大小不变</small>

图中每个大方块对应一个逻辑任务，`bx`是任务编号。M从64增大到192时，结果块从2个增加到6个。每个块的大小没有变化，每个任务所需的L1、L0C和UB空间也没有随M增大。

这里的任务数量不等于物理Cube核或Vector核的数量。代码描述需要处理哪些结果块，运行时再将这些任务分配给硬件执行。本节先只支持M为64的正整数倍，第4部分会解释这个限制。

**适用范围：本例固定 N=K=128，只支持动态 M。** 动态尺寸的声明和检查思路也可用于其他维度，但支持动态 N、K 还需调整相关索引、循环或边界处理，不能仅替换维度声明。


## 2. 完整kernel：在原来的计算上加入动态行数

下面保留完整kernel供对照，可按需展开。后文会逐段解释三个标记：①将固定M改为动态符号；②增加对行数的约束；③根据本次M确定任务数。③的表达式沿用5.2，但其中的M现在要到调用时才能确定。局部空间分配、矩阵乘和加偏置的计算主体保持不变。

In [2]:
%%writefile src/matmul_bias_dynamic.py
import tilelang
import tilelang.ascend.language as T
from tilelang.ascend.language import simd as S

tilelang.disable_cache()

N = K = 128
BM = BN = BK = 64
M = T.dynamic("num_rows")  # ① M是运行时行数

@T.prim_func
def matmul_bias_dynamic(
    A: T.Tensor((M, K), "float16"),
    B: T.Tensor((N, K), "float16"),
    Bias: T.Tensor((N,), "float32"),
    Y: T.Tensor((M, N), "float32"),
):
    T.assume(M > 0)  # ② 调用方保证的条件
    T.assume(M % BM == 0)

    # ③ 任务数根据本次M计算
    with T.Kernel((M // BM) * (N // BN)) as bx:
        row = bx // (N // BN)
        col = bx % (N // BN)

        a = T.alloc_l1((BM, BK), "float16")
        b = T.alloc_l1((BN, BK), "float16")
        acc = T.alloc_l0c((BM, BN), "float32")
        v = T.alloc_shared((BM // 2, BN), "float32")
        bias = T.alloc_shared((BN,), "float32")

        for ki in T.serial(K // BK):
            T.copy(A[row * BM:(row + 1) * BM,
                     ki * BK:(ki + 1) * BK], a)
            T.copy(B[col * BN:(col + 1) * BN,
                     ki * BK:(ki + 1) * BK], b)
            T.gemm(a, b, acc,
                   transpose_B=True, clear_accum=(ki == 0))

        T.dual_copy(acc, v)
        T.copy(Bias[col * BN:(col + 1) * BN], bias)

        with T.SimdVF():
            full = S.pset(32, "PAT_ALL")
            for i in T.serial(BM // 2):
                value = S.vld(v[i, 0])
                bv = S.vld(bias[0])
                result = S.vadd(value, bv, full)
                S.vsts(v[i, 0], result, full)

        T.dual_copy(v, Y[row * BM:(row + 1) * BM,
                         col * BN:(col + 1) * BN])


Overwriting src/matmul_bias_dynamic.py


<small>运行设置：<code>tilelang.disable_cache()</code>关闭的是编译缓存，不妨碍多次调用已经编译好的kernel对象。</small>

下面只展开动态M影响到的部分。

## 3. 顺着代码看：M怎样影响计算

### 声明动态行数，让A与Y使用同一个M

```python
M = T.dynamic("num_rows")

A: T.Tensor((M, K), "float16")
Y: T.Tensor((M, N), "float32")
```

`T.dynamic("num_rows")`创建一个表示整数尺寸的符号。`"num_rows"`是这个符号的名字。写下这行代码时，M还没有被固定为64、128或192。

两个`T.Tensor`声明让A和Y共享同一个行数：A有M行，Y也必须有M行。B和Bias的声明不含M，因此它们的形状保持不变。

Host端沿用5.2的`out_idx=-1`调用方式。新增的关键点是：接口从A的实际形状确定M，再创建相同行数的Y。例如，A为`(192, 128)`时，Y也为`(192, 128)`。

后面的Host循环使用小写`m`保存本次输入的实际行数。它是普通Python整数；这里的大写`M`是DSL中的动态符号。每次调用不需要重新定义M，也不需要另外传入一个名为`num_rows`的参数。

### 任务数随M变化，块编号的换算不变

```python
with T.Kernel((M // BM) * (N // BN)) as bx:
    row = bx // (N // BN)
    col = bx % (N // BN)
```

这三行的表达式与5.2相同。变化在于：M不再固定为128，**任务数`(M // BM) * (N // BN)`要根据本次输入的行数确定。**

列方向仍有`N // BN = 2`个块，因此继续用`row = bx // 2`和`col = bx % 2`换算块编号。随着M增大，沿行方向的块数和bx的取值范围一起增大：

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;">本次M</th>
<th style="text-align: left; vertical-align: top;">行方向块数</th>
<th style="text-align: left; vertical-align: top;">列方向块数</th>
<th style="text-align: left; vertical-align: top;">总任务数</th>
<th style="text-align: left; vertical-align: top;">bx范围</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;">64</td>
<td style="text-align: left; vertical-align: top;">1</td>
<td style="text-align: left; vertical-align: top;">2</td>
<td style="text-align: left; vertical-align: top;">2</td>
<td style="text-align: left; vertical-align: top;">0、1</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">128</td>
<td style="text-align: left; vertical-align: top;">2</td>
<td style="text-align: left; vertical-align: top;">2</td>
<td style="text-align: left; vertical-align: top;">4</td>
<td style="text-align: left; vertical-align: top;">0～3</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">192</td>
<td style="text-align: left; vertical-align: top;">3</td>
<td style="text-align: left; vertical-align: top;">2</td>
<td style="text-align: left; vertical-align: top;">6</td>
<td style="text-align: left; vertical-align: top;">0～5</td>
</tr>
</tbody>
</table>

例如，M=128时只需要bx=0～3；M=192时新增bx=4、5，负责Y最后64行的两个结果块。下面以新增的bx=5说明输入和输出的位置。

### 新增任务访问哪些行，原来的切片还适用吗？

M=192、bx=5时，`row=2`、`col=1`。读取A与写回Y仍使用原来的行范围表达式：

```python
row * BM : (row + 1) * BM
```

代入后就是`128:192`，即第128～191行。**切片表达式不需要改，变化的是row可以取到更大的值。**

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;">当前任务访问的数据</th>
<th style="text-align: left; vertical-align: top;">bx=5对应的范围</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;">A的输入行</td>
<td style="text-align: left; vertical-align: top;"><code>A[128:192, :]</code>，仍按BK分两轮读取K维</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">B的输入行</td>
<td style="text-align: left; vertical-align: top;"><code>B[64:128, :]</code>，由col决定</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>Bias</code></td>
<td style="text-align: left; vertical-align: top;"><code>Bias[64:128]</code>，由col决定</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">Y的输出块</td>
<td style="text-align: left; vertical-align: top;"><code>Y[128:192, 64:128]</code></td>
</tr>
</tbody>
</table>

B和Bias的选取不受新增行块的影响。新增任务只是为另一组A的行计算结果，并写到Y对应的行。

任务内部沿用5.2的实现：Cube沿K维累加，再由两个Vector侧加偏置并写回。因为BM、BN、BK没有变化，`acc`仍为64×64，每侧UB中的`v`仍为32×64；这些缓冲区的大小和相应计算语句都不需要修改。

## 4. 在调用前检查输入，再向编译器说明条件

完整kernel中有两条假设：

```python
T.assume(M > 0)
T.assume(M % BM == 0)
```

这两行将“行数为正、能够按BM整块划分”的条件告诉编译器。

> **先检查，再声明：** 调用方必须先保证条件成立。`T.assume`不会补齐输入，也不能代替Host侧的输入检查。

M变成动态值后，需要在每次调用前检查实际行数。下面的Host函数负责这一步；检查通过后才将A传到NPU并调用kernel。

In [3]:
%%writefile -a src/matmul_bias_dynamic.py

import torch
import torch_npu

def check_inputs(a, b, bias):
    if a.ndim != 2 or b.shape != (N, K) or bias.shape != (N,):
        raise ValueError("输入形状不符合要求")
    if a.shape[1] != K:
        raise ValueError("A的列数必须等于K")
    m = a.shape[0]
    if m <= 0 or m % BM != 0:
        raise ValueError("M必须为64的正整数倍")
    if a.dtype != torch.float16 or b.dtype != torch.float16 or bias.dtype != torch.float32:
        raise ValueError("A、B必须为float16，Bias必须为float32")
    if not all(x.is_contiguous() for x in (a, b, bias)):
        raise ValueError("输入必须连续存放")
    return m


Appending to src/matmul_bias_dynamic.py


重点看`m = a.shape[0]`和紧接着的判断：每次从A读取实际行数，再检查它是否为64的正整数倍。形状、数据类型和连续性仍需满足当前kernel的要求，完整代码中也保留了这些检查。

### 为什么当前示例拒绝 M=193？

**M 必须是64的正整数倍，是本节为简化教学而限定的输入范围。** 为了集中讲清动态维度的声明、任务数量的计算和编译结果的复用，本例只处理完整的64行块，暂不实现尾块处理。Matmul + Bias 这一计算本身并不要求 M 是64的整数倍。

M=193时，前192行能组成三个64行块，最后还剩一行。当前任务数使用`M // BM`，只会为前三个行块分配任务，最后一行没有对应的计算。如果只把任务数改成向上取整，又会产生一个不满64行的块，不能继续假定其中所有位置都有效。

要让实现支持M=193，需要补充尾块处理，并保证读入、计算和写回都正确处理块中的无效位置。在完成这些修改前，当前示例通过`check_inputs`在调用前抛出`ValueError("M必须为64的正整数倍")`，防止不满足实现条件的输入进入kernel。仅删除检查或修改`T.assume`，并不会自动补上尾块处理。


## 5. 编译一次，依次调用三种行数

第2部分定义了kernel，第4部分定义了输入检查函数。下面接着给出Host调用代码，它们按顺序组成完整程序。

![同一个编译后kernel依次接收三种行数的输入](images/05_03_compile_reuse.png)

<small>图5-4　编译一次，依次处理三种行数</small>

图中三条调用路径按顺序执行，并且都使用中间的同一个kernel对象。关键是把`tilelang.compile`放在循环外：先编译，再在每次循环中准备不同形状的A。

**追加Host调用代码**

<small>本段接在kernel定义和输入检查函数之后。</small>

In [4]:
%%writefile -a src/matmul_bias_dynamic.py
torch.npu.set_device(0)
kernel = tilelang.compile(matmul_bias_dynamic, target="ascend", out_idx=-1)

b_cpu = torch.ones((N, K), dtype=torch.float16)
bias_cpu = torch.arange(N, dtype=torch.float32)
b_npu, bias_npu = b_cpu.npu(), bias_cpu.npu()

for m in (64, 128, 192):
    a_cpu = torch.ones((m, K), dtype=torch.float16)
    check_inputs(a_cpu, b_cpu, bias_cpu)
    actual = kernel(a_cpu.npu(), b_npu, bias_npu).cpu()
    expected = a_cpu.float() @ b_cpu.float().T + bias_cpu
    torch.testing.assert_close(actual, expected, rtol=0.002, atol=0.002)
    print(f"M={m}，输出形状={tuple(actual.shape)}，完整输出比较通过")
    print("第0行的第0、63、64、127列：", actual[0, [0, 63, 64, 127]].tolist())

print("Verification passed!")


Appending to src/matmul_bias_dynamic.py


与5.2相比，这里重点看三处：

1. `tilelang.compile`放在循环外，只执行一次；三次调用都使用它返回的`kernel`对象。
2. B和Bias不随M变化，所以在循环前准备并传到设备。
3. 循环内按当前`m`创建A，先检查输入，再调用同一个`kernel`。接口从A的形状确定本次M，并返回相同行数的Y。

<small>前节参照：设备选择、输入输出搬运和完整比较的用法沿用5.2。</small>

**运行完整程序**

在本章目录的终端中运行，或执行下方Notebook单元：

```bash
python src/matmul_bias_dynamic.py
```

In [5]:
run_result = run_and_show([sys.executable, "-u", 'src/matmul_bias_dynamic.py'])


Loading tilelang libs from dev root: <已省略安装路径>
[W918 20:20:48.588909528 FunctionLoader.cpp:48] Warning: LD_PRELOAD detected, FunctionLoader prefers RTLD_DEFAULT for symbol resolution. (function operator())
2026-09-18 20:20:55  [TileLang:tilelang.jit.kernel:INFO] (kernel.py:131): TileLang begins to compile kernel `matmul_bias_dynamic` with `out_idx=-1`
2026-09-18 20:21:07  [TileLang:tilelang.jit.kernel:INFO] (kernel.py:139): TileLang completes to compile kernel `matmul_bias_dynamic`
M=64，输出形状=(64, 128)，完整输出比较通过
第0行的第0、63、64、127列： [128.0, 191.0, 192.0, 255.0]
M=128，输出形状=(128, 128)，完整输出比较通过
第0行的第0、63、64、127列： [128.0, 191.0, 192.0, 255.0]
M=192，输出形状=(192, 128)，完整输出比较通过
第0行的第0、63、64、127列： [128.0, 191.0, 192.0, 255.0]
Verification passed!


**怎样读三组输出**

输入构造方式与5.2相同：A、B全为1，Bias按列递增，所以第j列仍得到`128+j`。增加相同的输入行不会改变打印的四个数值；这里要关注的是Y的行数随M变化，而且每次都检查了完整的`M×128`个输出元素。

**编译复用的判断：** `tilelang.compile`应在循环外只执行一次，循环内始终调用同一个 `kernel` 对象。三组数值正确，并不能单独证明实现了编译复用。

## 6. 练习与解析

<small>练习建议：先对照代码作答，再展开解析。</small>

1. 如果下一次传入M=256的A，需要多少个逻辑任务？bx=7负责Y的哪一块？
2. 为什么M从64增大到192后，每个Vector侧的`v`仍然是`(32, 64)`？
3. 去掉`check_inputs`中“必须整除64”的检查，能否让这个kernel正确支持M=193？
4. 如果把`tilelang.compile`移到`for m in ...`循环中，即使结果正确，是否仍然完成了本节“一次编译、多次调用”的目标？

<details>
<summary>展开练习解析</summary>

**第1题：** 行方向有`256 // 64 = 4`块，列方向有`128 // 64 = 2`块，共8个任务。bx=7时，`row=7 // 2=3`、`col=7 % 2=1`，所以负责`Y[192:256, 64:128]`。

**第2题：** 每个任务仍然计算一个64×64结果块，并由两个Vector侧分别处理32行。M增大使任务数量增加，但BM和BN没有变化，所以每侧UB中的`v`仍是32×64。

**第3题：** 不能。整除限制来自当前教学示例省略了尾块处理；去掉检查并没有增加最后一行所需的任务，也没有处理尾块中的无效位置。补充这些逻辑后可以扩展输入范围，`T.assume`本身不会自动实现这些功能。

**第4题：** 没有。这样会在每轮循环执行编译调用，而本节要求在循环前编译一次，再重复调用它返回的对象。输入形状和计算结果都正确，与是否复用同一个编译结果，是两件需要分别检查的事。

</details>

<details>
<summary>选读：如果输入不是连续存放的</summary>

本节使用连续张量。例如，A的一行包含128个元素，相邻两行起点相隔128个元素。这个距离称为行步长。如果A来自某些切片或转置操作，实际行步长可能不同，不能继续按本例的连续布局处理。

TileLang还提供`T.StridedTensor`描述带步长的张量。下面只展示参数声明形式，其中`stride_m`表示行步长，不是对上面kernel的直接替换：

```python
A: T.StridedTensor(shape=(M, K), strides=(stride_m, 1), dtype="float16")
```

要支持这类输入，还需要在程序中声明步长，并检查相应搬运操作的要求。本节的完整程序仍要求输入连续存放；掌握动态行数后，再学习步长即可。

</details>

<small>相关文件：[动态行数完整程序](src/matmul_bias_dynamic.py)。</small>

[章节导航](README.md) · [上一节](05.02_matmul_bias.ipynb) · [下一节](05.04_practice.ipynb)